In [ ]:

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

SEED = 17
rng = np.random.default_rng(SEED)


## Section 0 - Generate synthetic sample data

In [ ]:

# --------------------------------
# Core universe and daily panel
# --------------------------------
n_tickers = 160
dates = pd.bdate_range("2019-01-01", "2023-12-29", freq="B")
tickers = [f"EQ{i:03d}" for i in range(n_tickers)]
sectors = ["Tech", "Health", "Financials", "Industrials", "Consumer", "Energy", "Utilities", "Materials", "Comm", "RealEstate"]

meta = pd.DataFrame({
    "ticker": tickers,
    "sector": rng.choice(sectors, size=n_tickers),
    "country": rng.choice(["US", "CA", "UK"], size=n_tickers, p=[0.80, 0.10, 0.10]),
    "shares_outstanding_m": rng.uniform(25, 2200, size=n_tickers),
    "base_beta": rng.normal(1.0, 0.20, size=n_tickers),
    "quality_tilt": rng.normal(0, 1, size=n_tickers),
    "value_tilt": rng.normal(0, 1, size=n_tickers),
    "growth_tilt": rng.normal(0, 1, size=n_tickers),
    "sentiment_tilt": rng.normal(0, 1, size=n_tickers),
    "liquidity_tilt": rng.normal(0, 1, size=n_tickers),
})
meta["industry"] = meta["sector"] + "_" + rng.integers(1, 5, size=n_tickers).astype(str)

market_ret = pd.Series(rng.normal(0.0002, 0.010, len(dates)), index=dates, name="mkt_ret")
sector_ret = pd.DataFrame(rng.normal(0, 0.005, (len(dates), len(sectors))), index=dates, columns=sectors)
regime = pd.Series(np.where(dates < "2020-03-01", 0, np.where(dates < "2020-09-01", 1, np.where(dates < "2022-01-01", 2, 3))), index=dates)
regime_scale = regime.map({0: 1.0, 1: 1.9, 2: 0.85, 3: 1.05}).values
market_ret = market_ret * regime_scale

rows = []
for _, row in meta.iterrows():
    n = len(dates)
    eps = rng.normal(0, 0.017, n)
    trend = pd.Series(rng.normal(0, 0.0011, n), index=dates).rolling(20, min_periods=1).mean()
    value = 0.00010 * row["value_tilt"]
    quality = 0.00008 * row["quality_tilt"]
    growth = 0.00010 * row["growth_tilt"]
    sentiment = 0.00006 * row["sentiment_tilt"]
    ret = (
        row["base_beta"] * market_ret.values
        + sector_ret[row["sector"]].values
        + trend.values
        + value + quality + growth + sentiment
        + eps
    )
    event_mask = rng.random(n) < 0.0035
    ret[event_mask] += rng.normal(0, 0.07, event_mask.sum())

    start_px = rng.uniform(8, 140)
    close = start_px * np.exp(np.cumsum(np.log1p(np.clip(ret, -0.95, None))))
    close = np.maximum(close, 0.5)

    volume = rng.lognormal(mean=13.2 + 0.2 * row["liquidity_tilt"], sigma=0.60, size=n) * (1 + 13 * np.abs(ret))
    volume = np.maximum(volume, 1000)
    spread_bps = np.maximum(2, 25 - 4 * row["liquidity_tilt"] + 20 * rng.random(n))
    borrow_bps_annual = np.maximum(0, 50 + 60 * np.maximum(-row["sentiment_tilt"], 0) + 150 * rng.random(n))

    rows.append(pd.DataFrame({
        "date": dates,
        "ticker": row["ticker"],
        "ret_1d": ret,
        "close": close,
        "volume": volume,
        "spread_bps": spread_bps,
        "borrow_bps_annual": borrow_bps_annual,
    }))

daily = pd.concat(rows, ignore_index=True).sort_values(["date", "ticker"]).reset_index(drop=True)
daily["dollar_volume"] = daily["close"] * daily["volume"]
daily = daily.merge(meta, on="ticker", how="left")
daily["market_cap_m"] = daily["close"] * daily["shares_outstanding_m"]

# Add a synthetic alpha scaffold
g = daily.groupby("ticker")
daily["mom_21d"] = g["ret_1d"].transform(lambda s: (1 + s).shift(1).rolling(21, min_periods=21).apply(np.prod, raw=True) - 1)
daily["vol_21d"] = g["ret_1d"].transform(lambda s: s.shift(1).rolling(21, min_periods=21).std())
daily["adv_21d"] = g["dollar_volume"].transform(lambda s: s.shift(1).rolling(21, min_periods=21).mean())
daily["log_adv_21d"] = np.log(daily["adv_21d"])
daily["ret_fwd_1d"] = g["ret_1d"].shift(-1)
daily["ret_fwd_5d"] = g["ret_1d"].transform(lambda s: (1 + s).shift(-1).rolling(5, min_periods=5).apply(np.prod, raw=True) - 1)

# cross-sectional zscores
for col in ["mom_21d", "vol_21d", "log_adv_21d"]:
    daily[f"{col}_z"] = daily.groupby("date")[col].transform(
        lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else 0.0
    )

daily["alpha_score"] = daily[["mom_21d_z", "log_adv_21d_z"]].mean(axis=1) - 0.5 * daily["vol_21d_z"]

daily.head()


In [ ]:

# --------------------------------
# Quarterly realized EPS events
# --------------------------------
quarter_ends = pd.date_range("2018-12-31", "2023-12-31", freq="Q")
eps_rows = []

for _, row in meta.iterrows():
    level = rng.uniform(0.10, 5.00)
    growth = 0.01 + 0.02 * row["growth_tilt"]

    for q in quarter_ends:
        announce_lag = int(rng.integers(18, 45))
        announce_date = pd.bdate_range(q, periods=announce_lag + 1)[-1]
        level = max(-2, level * (1 + rng.normal(growth, 0.08)))
        actual_eps = level + rng.normal(0, 0.12)
        eps_rows.append({
            "ticker": row["ticker"],
            "fiscal_q_end": q,
            "earnings_announce_date": announce_date,
            "actual_eps": actual_eps,
        })

actual_eps = pd.DataFrame(eps_rows).sort_values(["ticker", "earnings_announce_date"]).reset_index(drop=True)
actual_eps.head()


In [ ]:

# --------------------------------
# Analyst-level EPS estimate detail
# --------------------------------
brokers = [f"BRK{i:02d}" for i in range(1, 26)]
est_rows = []

for _, row in meta.iterrows():
    base_follow = int(rng.integers(6, 18))
    covered_brokers = rng.choice(brokers, size=base_follow, replace=False)
    actual_sub = actual_eps.loc[actual_eps["ticker"] == row["ticker"]].copy()

    for _, e in actual_sub.iterrows():
        q = e["fiscal_q_end"]
        ann_dt = e["earnings_announce_date"]
        actual = e["actual_eps"]

        start_date = q - pd.Timedelta(days=140)
        valid_est_dates = pd.bdate_range(start_date, ann_dt - pd.Timedelta(days=1), freq="B")

        base_consensus = actual + rng.normal(0, 0.18)

        for brk in covered_brokers:
            n_updates = int(rng.integers(2, 10))
            update_dates = np.sort(rng.choice(valid_est_dates, size=n_updates, replace=False))
            broker_bias = rng.normal(0, 0.10)
            est = base_consensus + broker_bias

            for ud in update_dates:
                revision = rng.normal(0.0 + 0.01 * row["sentiment_tilt"], 0.06)
                est = est + revision
                est_rows.append({
                    "ticker": row["ticker"],
                    "broker_id": brk,
                    "metric": "EPS",
                    "fiscal_q_end": q,
                    "estimate_date": pd.Timestamp(ud),
                    "estimate_value": est,
                    "revision_type": "UP" if revision > 0 else "DOWN",
                })

analyst_detail = pd.DataFrame(est_rows).sort_values(
    ["ticker", "fiscal_q_end", "broker_id", "estimate_date"]
).reset_index(drop=True)

analyst_detail.head()


In [ ]:

# Build a monthly rebalance snapshot for portfolio / backtest drills
daily["month"] = daily["date"].dt.to_period("M")
rebalance = (
    daily.sort_values(["ticker", "date"])
    .groupby(["ticker", "month"], as_index=False)
    .tail(1)
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

rebalance.head()



## Section 1 - Portfolio construction from signals

Focus areas:
- converting scores to ranks, buckets, and weights
- sector-neutral and beta-aware weighting
- liquidity / concentration / turnover constraints
- tradability filters
- portfolio diagnostics


### Drill 1.1 - Tradable universe filter


From `rebalance`, create `rebal_univ` containing only rows that satisfy:
- `close >= 5`
- `market_cap_m >= 500`
- `adv_21d >= 5e6`
- non-null `alpha_score`
- non-null `ret_fwd_5d`


In [ ]:

# TODO: create rebal_univ

# SOLUTION:
# rebal_univ = rebalance.loc[
#     (rebalance["close"] >= 5)
#     & (rebalance["market_cap_m"] >= 500)
#     & (rebalance["adv_21d"] >= 5e6)
#     & rebalance["alpha_score"].notna()
#     & rebalance["ret_fwd_5d"].notna()
# ].copy()


### Step 1.2 - Cross-sectional rank and bucket portfolios


On each rebalance date:
- create `alpha_rank_pct`
- assign quintiles using `alpha_score`
- assign deciles using `alpha_score`

Store as:
- `alpha_rank_pct`
- `alpha_quintile`
- `alpha_decile`


In [ ]:

# TODO: create alpha_rank_pct, alpha_quintile, alpha_decile

# SOLUTION:
# rebal_univ["alpha_rank_pct"] = rebal_univ.groupby("date")["alpha_score"].rank(pct=True, method="average")
# rebal_univ["alpha_quintile"] = rebal_univ.groupby("date")["alpha_score"].transform(
#     lambda s: pd.qcut(s.rank(method="first"), 5, labels=False, duplicates="drop") + 1
# )
# rebal_univ["alpha_decile"] = rebal_univ.groupby("date")["alpha_score"].transform(
#     lambda s: pd.qcut(s.rank(method="first"), 10, labels=False, duplicates="drop") + 1
# )


### Step 1.3 - Equal-weight top-minus-bottom portfolio


Construct daily rebalance weights for a quintile spread:
- long top quintile
- short bottom quintile
- equal weight within each side
- gross exposure = 1
- net exposure = 0

Store as `w_q5_q1_eq`.


In [ ]:

# TODO: create w_q5_q1_eq

# SOLUTION:
# long_mask = rebal_univ["alpha_quintile"] == rebal_univ.groupby("date")["alpha_quintile"].transform("max")
# short_mask = rebal_univ["alpha_quintile"] == rebal_univ.groupby("date")["alpha_quintile"].transform("min")
# n_long = long_mask.groupby(rebal_univ["date"]).transform("sum")
# n_short = short_mask.groupby(rebal_univ["date"]).transform("sum")
# rebal_univ["w_q5_q1_eq"] = 0.0
# rebal_univ.loc[long_mask, "w_q5_q1_eq"] = 0.5 / n_long[long_mask]
# rebal_univ.loc[short_mask, "w_q5_q1_eq"] = -0.5 / n_short[short_mask]


### Step 1.4 - Score-proportional market-neutral portfolio


Create a portfolio from `alpha_score`:
1. demean score by date
2. divide by sum(abs(score_demeaned))
3. store as `w_score_neutral`


In [ ]:

# TODO: create w_score_neutral

# SOLUTION:
# score_dm = rebal_univ["alpha_score"] - rebal_univ.groupby("date")["alpha_score"].transform("mean")
# denom = score_dm.groupby(rebal_univ["date"]).transform(lambda s: s.abs().sum())
# rebal_univ["w_score_neutral"] = score_dm / denom


### Step 1.5 - Sector-neutral portfolio weights


Create sector-neutral score-based weights:
- demean `alpha_score` within each `(date, sector)`
- normalize within date to gross 1
- store as `w_sector_neutral`

This should reduce sector bets relative to a plain score-neutral portfolio.


In [ ]:

# TODO: create w_sector_neutral

# SOLUTION:
# sector_dm = rebal_univ["alpha_score"] - rebal_univ.groupby(["date", "sector"])["alpha_score"].transform("mean")
# denom = sector_dm.groupby(rebal_univ["date"]).transform(lambda s: s.abs().sum())
# rebal_univ["w_sector_neutral"] = sector_dm / denom


### Step 1.6 - Liquidity-capped weights


Starting from `w_score_neutral`, cap each absolute position by:
- `max_name_abs = 0.03`
- `max_participation_of_adv = 0.10` of 21-day ADV, expressed as a weight proxy

For the exercise, approximate an ADV-based weight cap with:
`max_weight_adv = 0.10 * adv_21d / sum(adv_21d) by date`

Then clip and renormalize to gross 1.
Store as `w_liq_capped`.


In [ ]:

# TODO: create w_liq_capped

# SOLUTION:
# max_name_abs = 0.03
# adv_weight_cap = 0.10 * rebal_univ["adv_21d"] / rebal_univ.groupby("date")["adv_21d"].transform("sum")
# abs_cap = np.minimum(max_name_abs, adv_weight_cap)
# w = rebal_univ["w_score_neutral"].clip(lower=-abs_cap, upper=abs_cap)
# gross = w.groupby(rebal_univ["date"]).transform(lambda s: s.abs().sum())
# rebal_univ["w_liq_capped"] = w / gross


### Step 1.7 - Beta-neutral overlay


Approximate beta-neutralization using `base_beta`.

Starting from `w_score_neutral`, adjust weights each date so that:
- net beta exposure is approximately zero
- preserve roughly the same gross after renormalization

Store as `w_beta_neutral`.

Hint:
- Compute daily beta exposure = sum(w * beta)
- Remove a multiple of beta from the weights cross-section
- Renormalize


In [ ]:

# TODO: create w_beta_neutral

# SOLUTION:
# def beta_neutralize_one_day(df, w_col="w_score_neutral", beta_col="base_beta"):
#     out = df[w_col].copy()
#     beta = df[beta_col].astype(float)
#     valid = out.notna() & beta.notna()
#     if valid.sum() < 3:
#         return out
#     w = out.loc[valid].to_numpy()
#     b = beta.loc[valid].to_numpy()
#     lam = np.dot(w, b) / np.dot(b, b)
#     w_adj = w - lam * b
#     gross = np.abs(w_adj).sum()
#     if gross > 0:
#         w_adj = w_adj / gross
#     out.loc[valid] = w_adj
#     return out
#
# rebal_univ["w_beta_neutral"] = (
#     rebal_univ.groupby("date", group_keys=False)
#     .apply(beta_neutralize_one_day)
# )


### Step 1.8 - Turnover-constrained target blending


Create a turnover-aware target by blending prior weights and today's desired weights:

`w_blend = 0.6 * w_prev + 0.4 * w_score_neutral`

Then renormalize to gross 1 by date.

Store as:
- `w_prev`
- `w_blend`


In [ ]:

# TODO: create w_prev and w_blend

# SOLUTION:
# rebal_univ = rebal_univ.sort_values(["ticker", "date"]).copy()
# rebal_univ["w_prev"] = rebal_univ.groupby("ticker")["w_score_neutral"].shift(1).fillna(0)
# w_blend = 0.6 * rebal_univ["w_prev"] + 0.4 * rebal_univ["w_score_neutral"]
# gross = w_blend.groupby(rebal_univ["date"]).transform(lambda s: s.abs().sum())
# rebal_univ["w_blend"] = w_blend / gross


### Step 1.9 - Portfolio diagnostics by date


Build a daily diagnostics DataFrame `port_diag_inputs` for one chosen weight column, e.g. `w_liq_capped`, with:
- gross
- net
- effective_n = 1 / sum(w^2)
- max_long
- max_short
- beta_exposure = sum(w * base_beta)
- sector concentration = max absolute sector weight

Output one row per date.


In [ ]:

# TODO: create port_diag_inputs for w_liq_capped

# SOLUTION:
# wcol = "w_liq_capped"
# tmp = rebal_univ.copy()
# tmp["w_sector"] = tmp.groupby(["date", "sector"])[wcol].transform("sum")
# port_diag_inputs = tmp.groupby("date").apply(
#     lambda df: pd.Series({
#         "gross": df[wcol].abs().sum(),
#         "net": df[wcol].sum(),
#         "effective_n": 1 / np.sum(np.square(df[wcol])) if np.sum(np.square(df[wcol])) > 0 else np.nan,
#         "max_long": df[wcol].max(),
#         "max_short": df[wcol].min(),
#         "beta_exposure": np.sum(df[wcol] * df["base_beta"]),
#         "sector_concentration": df.groupby("sector")[wcol].sum().abs().max(),
#     })
# ).reset_index()



## Section 2 - Backtest implementation

Focus areas:
- holding-period return calculation
- transaction costs
- turnover
- long / short PnL decomposition
- decile spread evaluation
- walk-forward style diagnostics


### Step 2.1 - One-period rebalance backtest


Using `rebal_univ` and `w_score_neutral`, compute a monthly rebalance backtest:
- portfolio return = sum(weight * ret_fwd_5d) or next-period realized return proxy
- one row per rebalance date
- output `bt_basic`


In [ ]:

# TODO: create bt_basic using w_score_neutral and ret_fwd_5d

# SOLUTION:
# bt_basic = (
#     rebal_univ.groupby("date")
#     .apply(lambda df: pd.Series({
#         "port_ret": np.nansum(df["w_score_neutral"] * df["ret_fwd_5d"]),
#         "gross": df["w_score_neutral"].abs().sum(),
#         "net": df["w_score_neutral"].sum(),
#     }))
#     .reset_index()
# )


### Step 2.2 - Transaction-cost-adjusted backtest


Build a cost-adjusted backtest using `w_liq_capped`.

Approximate one-way trading cost per rebalance as:
- spread cost: `0.5 * spread_bps / 10000 * abs(delta_w)`
- borrow cost on short book over 5 days:
  `borrow_bps_annual / 10000 * 5/252 * abs(short_weight)`
- total portfolio return net of both costs

Output `bt_costed` with:
- gross_ret
- spread_cost
- borrow_cost
- net_ret
- turnover_1way


In [ ]:

# TODO: create bt_costed

# SOLUTION:
# tmp = rebal_univ.sort_values(["ticker", "date"]).copy()
# tmp["w_prev_liq"] = tmp.groupby("ticker")["w_liq_capped"].shift(1).fillna(0.0)
# tmp["delta_w"] = tmp["w_liq_capped"] - tmp["w_prev_liq"]
# tmp["spread_cost_name"] = 0.5 * (tmp["spread_bps"] / 10000.0) * tmp["delta_w"].abs()
# tmp["borrow_cost_name"] = (tmp["borrow_bps_annual"] / 10000.0) * (5/252) * tmp["w_liq_capped"].clip(upper=0).abs()
#
# bt_costed = tmp.groupby("date").apply(
#     lambda df: pd.Series({
#         "gross_ret": np.nansum(df["w_liq_capped"] * df["ret_fwd_5d"]),
#         "spread_cost": df["spread_cost_name"].sum(),
#         "borrow_cost": df["borrow_cost_name"].sum(),
#         "net_ret": np.nansum(df["w_liq_capped"] * df["ret_fwd_5d"]) - df["spread_cost_name"].sum() - df["borrow_cost_name"].sum(),
#         "turnover_1way": 0.5 * df["delta_w"].abs().sum(),
#     })
# ).reset_index()


### Step 2.3 - Top decile minus bottom decile spread backtest


Using `alpha_decile`, compute the decile spread time series:
- equal-weight top decile forward return
- equal-weight bottom decile forward return
- top-minus-bottom spread
- cumulative spread index


In [ ]:

# TODO: create decile_bt

# SOLUTION:
# dec = (
#     rebal_univ.loc[rebal_univ["alpha_decile"].isin([1, 10])]
#     .groupby(["date", "alpha_decile"])["ret_fwd_5d"]
#     .mean()
#     .unstack()
# )
# dec["spread"] = dec[10] - dec[1]
# dec["cum_spread"] = (1 + dec["spread"].fillna(0)).cumprod()
# decile_bt = dec.reset_index()


### Step 2.4 - Long and short side attribution


For `w_beta_neutral`, compute by date:
- total portfolio return
- contribution from longs
- contribution from shorts
- weighted average alpha score in longs
- weighted average alpha score in shorts

Output `side_attr`.


In [ ]:

# TODO: create side_attr

# SOLUTION:
# def side_attr_one_day(df):
#     w = df["w_beta_neutral"].fillna(0)
#     long_w = w.clip(lower=0)
#     short_w = w.clip(upper=0)
#     return pd.Series({
#         "port_ret": np.sum(w * df["ret_fwd_5d"]),
#         "long_contrib": np.sum(long_w * df["ret_fwd_5d"]),
#         "short_contrib": np.sum(short_w * df["ret_fwd_5d"]),
#         "long_alpha_avg": np.sum(long_w * df["alpha_score"]) / long_w.sum() if long_w.sum() > 0 else np.nan,
#         "short_alpha_avg": np.sum(short_w.abs() * df["alpha_score"]) / short_w.abs().sum() if short_w.abs().sum() > 0 else np.nan,
#     })
#
# side_attr = rebal_univ.groupby("date").apply(side_attr_one_day).reset_index()


### Step 2.5 - Performance summary statistics


For any backtest return series, compute:
- mean return
- std dev
- annualized return
- annualized vol
- Sharpe
- hit rate
- max drawdown
- average turnover (if available)

Create a helper function:
`performance_summary(return_series, periods_per_year=12)`


In [ ]:

# TODO: write performance_summary

# SOLUTION:
# def performance_summary(return_series, periods_per_year=12, turnover_series=None):
#     r = pd.Series(return_series).dropna()
#     ann_ret = (1 + r).prod() ** (periods_per_year / len(r)) - 1 if len(r) > 0 else np.nan
#     ann_vol = r.std(ddof=1) * np.sqrt(periods_per_year) if len(r) > 1 else np.nan
#     sharpe = ann_ret / ann_vol if ann_vol and ann_vol > 0 else np.nan
#     wealth = (1 + r).cumprod()
#     drawdown = wealth / wealth.cummax() - 1
#     return pd.Series({
#         "mean": r.mean(),
#         "std": r.std(ddof=1),
#         "ann_return": ann_ret,
#         "ann_vol": ann_vol,
#         "sharpe": sharpe,
#         "hit_rate": (r > 0).mean(),
#         "max_drawdown": drawdown.min() if len(drawdown) else np.nan,
#         "avg_turnover": pd.Series(turnover_series).dropna().mean() if turnover_series is not None else np.nan,
#     })


### Step 2.6 - Rolling out-of-sample diagnostics


Using a backtest series, compute rolling 12-period:
- mean
- Sharpe
- hit rate

Return a DataFrame with one row per date.


In [ ]:

# TODO: compute rolling 12-period diagnostics on bt_costed['net_ret']

# SOLUTION:
# tmp = bt_costed.set_index("date")["net_ret"]
# rolling_diag = pd.DataFrame({
#     "roll_mean_12": tmp.rolling(12).mean(),
#     "roll_sharpe_12": tmp.rolling(12).mean() / tmp.rolling(12).std(ddof=1) * np.sqrt(12),
#     "roll_hit_12": tmp.rolling(12).apply(lambda x: np.mean(np.array(x) > 0), raw=True),
# }).reset_index()


### Step 2.7 - Capacity / trading-friction stress test


Stress test the backtest by varying:
- spread cost multiplier in {0.5x, 1.0x, 2.0x}
- borrow cost multiplier in {0.5x, 1.0x, 2.0x}

Produce a summary table of annualized return and Sharpe under each scenario.


In [ ]:

# TODO: create stress_summary

# SOLUTION:
# scenarios = []
# for sm in [0.5, 1.0, 2.0]:
#     for bm in [0.5, 1.0, 2.0]:
#         net_r = bt_costed["gross_ret"] - sm * bt_costed["spread_cost"] - bm * bt_costed["borrow_cost"]
#         summ = performance_summary(net_r, periods_per_year=12)
#         scenarios.append({
#             "spread_mult": sm,
#             "borrow_mult": bm,
#             "ann_return": summ["ann_return"],
#             "sharpe": summ["sharpe"],
#         })
# stress_summary = pd.DataFrame(scenarios)



## Section 3 - Statistics / econometrics in code

Focus areas:
- cross-sectional IC
- t-stats
- regressions
- factor neutralization
- bootstrap inference
- monotonicity and bucket tests
- panel regressions in interview-safe code


### Step 3.1 - Daily Pearson and Spearman IC


Compute daily cross-sectional ICs between:
- `alpha_score` and `ret_fwd_5d`

Return:
- `ic_ts`
- summary stats with mean IC, IC vol, IC IR, hit rate


In [ ]:

# TODO: create ic_ts and ic_summary

# SOLUTION:
# def ic_one_day(df):
#     return pd.Series({
#         "ic_pearson": df["alpha_score"].corr(df["ret_fwd_5d"], method="pearson"),
#         "ic_spearman": df["alpha_score"].corr(df["ret_fwd_5d"], method="spearman"),
#         "n": len(df),
#     })
#
# ic_ts = rebal_univ[["date", "alpha_score", "ret_fwd_5d"]].dropna().groupby("date").apply(ic_one_day).reset_index()
# ic_summary = pd.Series({
#     "mean_ic": ic_ts["ic_spearman"].mean(),
#     "ic_vol": ic_ts["ic_spearman"].std(ddof=1),
#     "ic_ir": ic_ts["ic_spearman"].mean() / ic_ts["ic_spearman"].std(ddof=1) if ic_ts["ic_spearman"].std(ddof=1) > 0 else np.nan,
#     "ic_hit_rate": (ic_ts["ic_spearman"] > 0).mean(),
# })


### Step 3.2 - IC t-stat


Compute a t-statistic for mean daily rank IC:
- `t = mean(IC) / (std(IC) / sqrt(T))`

Report:
- t-stat
- number of observations


In [ ]:

# TODO: compute rank_ic_tstat

# SOLUTION:
# rank_ic = ic_ts["ic_spearman"].dropna()
# rank_ic_tstat = pd.Series({
#     "t_stat": rank_ic.mean() / (rank_ic.std(ddof=1) / np.sqrt(len(rank_ic))) if len(rank_ic) > 1 else np.nan,
#     "n_obs": len(rank_ic),
# })


### Step 3.3 - Cross-sectional regression by date


For each date, run a cross-sectional regression:

`ret_fwd_5d ~ alpha_score + log_adv_21d_z + vol_21d_z`

Return a DataFrame with one row per date containing:
- intercept
- alpha_score coefficient
- log_adv_21d_z coefficient
- vol_21d_z coefficient
- R^2


In [ ]:

# TODO: create xsec_reg_ts

# SOLUTION:
# def ols_one_day(df):
#     cols = ["ret_fwd_5d", "alpha_score", "log_adv_21d_z", "vol_21d_z"]
#     tmp = df[cols].dropna()
#     if len(tmp) < 10:
#         return pd.Series({
#             "intercept": np.nan,
#             "beta_alpha": np.nan,
#             "beta_log_adv": np.nan,
#             "beta_vol": np.nan,
#             "r2": np.nan,
#         })
#     y = tmp["ret_fwd_5d"].to_numpy()
#     X = tmp[["alpha_score", "log_adv_21d_z", "vol_21d_z"]].to_numpy()
#     X = np.column_stack([np.ones(len(X)), X])
#     beta, *_ = np.linalg.lstsq(X, y, rcond=None)
#     yhat = X @ beta
#     ss_res = np.sum((y - yhat) ** 2)
#     ss_tot = np.sum((y - y.mean()) ** 2)
#     r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
#     return pd.Series({
#         "intercept": beta[0],
#         "beta_alpha": beta[1],
#         "beta_log_adv": beta[2],
#         "beta_vol": beta[3],
#         "r2": r2,
#     })
#
# xsec_reg_ts = rebal_univ.groupby("date").apply(ols_one_day).reset_index()


### Step 3.4 - Fama-MacBeth style average slope


Using the cross-sectional regression output from Drill 3.3, compute:
- average `beta_alpha`
- std of `beta_alpha`
- t-stat of `beta_alpha`

Store in `fm_alpha_summary`.


In [ ]:

# TODO: compute fm_alpha_summary

# SOLUTION:
# s = xsec_reg_ts["beta_alpha"].dropna()
# fm_alpha_summary = pd.Series({
#     "mean_beta_alpha": s.mean(),
#     "std_beta_alpha": s.std(ddof=1),
#     "tstat_beta_alpha": s.mean() / (s.std(ddof=1) / np.sqrt(len(s))) if len(s) > 1 else np.nan,
#     "n_dates": len(s),
# })


### Step 3.5 - Same-date factor neutralization via regression residuals


Residualize `alpha_score` each date against:
- sector dummies
- `log_adv_21d_z`
- `vol_21d_z`

Store residualized score as `alpha_neutralized`.

This is a very common interview task.


In [ ]:

# TODO: create alpha_neutralized

# SOLUTION:
# def neutralize_one_day(df):
#     out = pd.Series(np.nan, index=df.index)
#     tmp = df[["alpha_score", "log_adv_21d_z", "vol_21d_z", "sector"]].dropna().copy()
#     if len(tmp) < 15:
#         return out
#     dummies = pd.get_dummies(tmp["sector"], drop_first=True)
#     X = pd.concat([tmp[["log_adv_21d_z", "vol_21d_z"]], dummies], axis=1)
#     X = np.column_stack([np.ones(len(X)), X.to_numpy(dtype=float)])
#     y = tmp["alpha_score"].to_numpy(dtype=float)
#     beta, *_ = np.linalg.lstsq(X, y, rcond=None)
#     resid = y - X @ beta
#     out.loc[tmp.index] = resid
#     return out
#
# rebal_univ["alpha_neutralized"] = rebal_univ.groupby("date", group_keys=False).apply(neutralize_one_day)


### Step 3.6 - Monotonicity / bucket test


Using `alpha_decile`, compute for each decile:
- average forward return
- standard deviation of forward return
- hit rate
- average alpha score

Return `bucket_summary` ordered by decile.


In [ ]:

# TODO: create bucket_summary

# SOLUTION:
# bucket_summary = (
#     rebal_univ.groupby("alpha_decile")
#     .agg(
#         avg_fwd_ret=("ret_fwd_5d", "mean"),
#         std_fwd_ret=("ret_fwd_5d", "std"),
#         hit_rate=("ret_fwd_5d", lambda s: (s > 0).mean()),
#         avg_alpha=("alpha_score", "mean"),
#     )
#     .reset_index()
#     .sort_values("alpha_decile")
# )


### Step 3.7 - Bootstrap confidence interval for mean rank IC


Bootstrap the mean daily rank IC 1,000 times and estimate:
- bootstrap mean
- 5th percentile
- 95th percentile

Store results in `ic_bootstrap_summary`.


In [ ]:

# TODO: bootstrap the mean rank IC

# SOLUTION:
# rank_ic = ic_ts["ic_spearman"].dropna().to_numpy()
# boot = []
# for _ in range(1000):
#     sample = rng.choice(rank_ic, size=len(rank_ic), replace=True)
#     boot.append(sample.mean())
# boot = np.array(boot)
# ic_bootstrap_summary = pd.Series({
#     "boot_mean": boot.mean(),
#     "p05": np.quantile(boot, 0.05),
#     "p95": np.quantile(boot, 0.95),
# })


### Step 3.8 - Event-study style post-earnings drift diagnostic


Create a simple event study around earnings announcements:
- align each earnings event to the next available daily row
- compute average 5-day forward return after announcement
- bucket events by whether actual EPS beat the most recent consensus estimate

Output:
- event_summary with beat vs miss groups


In [ ]:

# TODO: create a beat/miss event summary using analyst consensus and actual EPS

# SOLUTION:
# # First derive latest consensus prior to announcement
# analyst_last = analyst_detail.sort_values(["ticker", "fiscal_q_end", "broker_id", "estimate_date"]).groupby(
#     ["ticker", "fiscal_q_end", "broker_id"], as_index=False
# ).tail(1)
# consensus = analyst_last.groupby(["ticker", "fiscal_q_end"], as_index=False)["estimate_value"].mean().rename(columns={"estimate_value": "consensus_eps"})
# eps_evt = actual_eps.merge(consensus, on=["ticker", "fiscal_q_end"], how="left")
# eps_evt["beat_flag"] = np.where(eps_evt["actual_eps"] >= eps_evt["consensus_eps"], "beat", "miss")
# evt_panel = pd.merge_asof(
#     eps_evt.sort_values(["ticker", "earnings_announce_date"]),
#     daily[["ticker", "date", "ret_fwd_5d"]].sort_values(["ticker", "date"]),
#     by="ticker",
#     left_on="earnings_announce_date",
#     right_on="date",
#     direction="forward",
# )
# event_summary = evt_panel.groupby("beat_flag")["ret_fwd_5d"].agg(["mean", "median", "count"]).reset_index()



## Section 4 - SQL drills for equity research workflows

These are intentionally practical and heavily focused on **analyst data** and **point-in-time research queries**.

Assume the following simplified tables exist:

### `prices_daily`
- `trade_date`
- `ticker`
- `close_price`
- `volume`
- `dollar_volume`
- `market_cap`
- `ret_1d`

### `security_master`
- `ticker`
- `perm_id`
- `sector`
- `industry`
- `country`
- `primary_exchange`
- `effective_from`
- `effective_to`

### `analyst_eps_detail`
- `ticker`
- `broker_id`
- `metric`  -- e.g. 'EPS'
- `fiscal_period_type` -- e.g. 'Q', 'A'
- `fiscal_q_end`
- `estimate_date`
- `estimate_value`
- `currency`
- `is_active`

### `earnings_actuals`
- `ticker`
- `fiscal_q_end`
- `announce_date`
- `actual_eps`
- `currency`

### `fundamentals_quarterly`
- `ticker`
- `fiscal_q_end`
- `announce_date`
- `sales`
- `assets`
- `book_equity`
- `net_income`

### SQL assumptions
- Use ANSI-ish SQL where possible.
- Window functions are fair game.
- If syntax differs slightly by warehouse, focus on the *logic*.


### SQL Step 4.1 - Latest broker-level EPS estimate as of a research date


For each `(ticker, broker_id, fiscal_q_end)`, pull the latest EPS estimate **known as of** `'2023-06-30'`.

You should return:
- ticker
- broker_id
- fiscal_q_end
- estimate_date
- estimate_value


In [ ]:

# TODO: write SQL

sql_4_1 = '''
-- SOLUTION:
-- SELECT ticker, broker_id, fiscal_q_end, estimate_date, estimate_value
-- FROM (
--     SELECT
--         ticker,
--         broker_id,
--         fiscal_q_end,
--         estimate_date,
--         estimate_value,
--         ROW_NUMBER() OVER (
--             PARTITION BY ticker, broker_id, fiscal_q_end
--             ORDER BY estimate_date DESC
--         ) AS rn
--     FROM analyst_eps_detail
--     WHERE metric = 'EPS'
--       AND fiscal_period_type = 'Q'
--       AND estimate_date <= DATE '2023-06-30'
--       AND is_active = 1
-- ) x
-- WHERE rn = 1;
'''


### SQL Step 4.2 - Point-in-time consensus EPS as of each month-end


For each month-end date in `prices_daily`, compute the **consensus EPS estimate** for the next fiscal quarter:
- use latest estimate per `(ticker, broker_id, fiscal_q_end)` as of that month-end
- then average across brokers

Return:
- trade_date
- ticker
- fiscal_q_end
- consensus_eps


In [ ]:

# TODO: write SQL

sql_4_2 = '''
-- SOLUTION:
-- WITH month_ends AS (
--     SELECT DISTINCT trade_date
--     FROM (
--         SELECT
--             trade_date,
--             ticker,
--             ROW_NUMBER() OVER (
--                 PARTITION BY ticker, DATE_TRUNC('month', trade_date)
--                 ORDER BY trade_date DESC
--             ) AS rn
--         FROM prices_daily
--     ) t
--     WHERE rn = 1
-- ),
-- broker_latest AS (
--     SELECT
--         m.trade_date,
--         a.ticker,
--         a.broker_id,
--         a.fiscal_q_end,
--         a.estimate_value,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, a.ticker, a.broker_id, a.fiscal_q_end
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM month_ends m
--     JOIN analyst_eps_detail a
--       ON a.estimate_date <= m.trade_date
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
-- ),
-- filtered AS (
--     SELECT *
--     FROM broker_latest
--     WHERE rn = 1
-- ),
-- next_q AS (
--     SELECT
--         trade_date,
--         ticker,
--         fiscal_q_end,
--         estimate_value,
--         ROW_NUMBER() OVER (
--             PARTITION BY trade_date, ticker
--             ORDER BY fiscal_q_end
--         ) AS fiscal_rank
--     FROM filtered
--     WHERE fiscal_q_end >= trade_date
-- )
-- SELECT
--     trade_date,
--     ticker,
--     fiscal_q_end,
--     AVG(estimate_value) AS consensus_eps
-- FROM next_q
-- WHERE fiscal_rank = 1
-- GROUP BY 1,2,3;
'''


### SQL Step 4.3 - 30-day EPS revision feature


As of each month-end and ticker, compute:
- current consensus EPS for next quarter
- consensus EPS from 30 calendar days earlier
- revision_30d = current - prior_30d

Return one row per `(trade_date, ticker)`.


In [ ]:

# TODO: write SQL

sql_4_3 = '''
-- SOLUTION:
-- WITH month_ends AS (
--     SELECT trade_date, ticker
--     FROM (
--         SELECT
--             trade_date,
--             ticker,
--             ROW_NUMBER() OVER (
--                 PARTITION BY ticker, DATE_TRUNC('month', trade_date)
--                 ORDER BY trade_date DESC
--             ) AS rn
--         FROM prices_daily
--     ) x
--     WHERE rn = 1
-- ),
-- broker_latest AS (
--     SELECT
--         m.trade_date,
--         m.ticker,
--         a.broker_id,
--         a.fiscal_q_end,
--         a.estimate_value,
--         a.estimate_date,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, m.ticker, a.broker_id, a.fiscal_q_end
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM month_ends m
--     JOIN analyst_eps_detail a
--       ON a.ticker = m.ticker
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
--      AND a.estimate_date <= m.trade_date
-- ),
-- current_cons AS (
--     SELECT trade_date, ticker, fiscal_q_end, AVG(estimate_value) AS current_consensus
--     FROM broker_latest
--     WHERE rn = 1
--     GROUP BY 1,2,3
-- ),
-- next_current AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY trade_date, ticker
--                 ORDER BY fiscal_q_end
--             ) AS fiscal_rank
--         FROM current_cons
--         WHERE fiscal_q_end >= trade_date
--     ) z
--     WHERE fiscal_rank = 1
-- ),
-- broker_latest_30 AS (
--     SELECT
--         m.trade_date,
--         m.ticker,
--         a.broker_id,
--         a.fiscal_q_end,
--         a.estimate_value,
--         a.estimate_date,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, m.ticker, a.broker_id, a.fiscal_q_end
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM month_ends m
--     JOIN analyst_eps_detail a
--       ON a.ticker = m.ticker
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
--      AND a.estimate_date <= m.trade_date - INTERVAL '30' DAY
-- ),
-- prior_cons AS (
--     SELECT trade_date, ticker, fiscal_q_end, AVG(estimate_value) AS prior_30d_consensus
--     FROM broker_latest_30
--     WHERE rn = 1
--     GROUP BY 1,2,3
-- ),
-- next_prior AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY trade_date, ticker
--                 ORDER BY fiscal_q_end
--             ) AS fiscal_rank
--         FROM prior_cons
--         WHERE fiscal_q_end >= trade_date
--     ) z
--     WHERE fiscal_rank = 1
-- )
-- SELECT
--     c.trade_date,
--     c.ticker,
--     c.fiscal_q_end,
--     c.current_consensus,
--     p.prior_30d_consensus,
--     c.current_consensus - p.prior_30d_consensus AS revision_30d
-- FROM next_current c
-- LEFT JOIN next_prior p
--   ON c.trade_date = p.trade_date
--  AND c.ticker = p.ticker
--  AND c.fiscal_q_end = p.fiscal_q_end;
'''


### SQL Step 4.4 - Net upward revision breadth by ticker/date


As of each month-end and ticker, compute for the next fiscal quarter:
- number of brokers with latest estimate revision up in the last 30 days
- number of brokers with latest estimate revision down in the last 30 days
- net revision breadth = (up - down) / total brokers

This is a common sell-side estimate feature.


In [ ]:

# TODO: write SQL

sql_4_4 = '''
-- SOLUTION:
-- WITH month_ends AS (
--     SELECT trade_date, ticker
--     FROM (
--         SELECT
--             trade_date,
--             ticker,
--             ROW_NUMBER() OVER (
--                 PARTITION BY ticker, DATE_TRUNC('month', trade_date)
--                 ORDER BY trade_date DESC
--             ) AS rn
--         FROM prices_daily
--     ) x
--     WHERE rn = 1
-- ),
-- latest_est AS (
--     SELECT
--         m.trade_date,
--         m.ticker,
--         a.broker_id,
--         a.fiscal_q_end,
--         a.estimate_date,
--         a.revision_type,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, m.ticker, a.broker_id, a.fiscal_q_end
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM month_ends m
--     JOIN analyst_eps_detail a
--       ON a.ticker = m.ticker
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
--      AND a.estimate_date <= m.trade_date
--      AND a.estimate_date > m.trade_date - INTERVAL '30' DAY
-- ),
-- next_q AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY trade_date, ticker, broker_id
--                 ORDER BY fiscal_q_end
--             ) AS fiscal_rank
--         FROM latest_est
--         WHERE rn = 1
--           AND fiscal_q_end >= trade_date
--     ) z
--     WHERE fiscal_rank = 1
-- )
-- SELECT
--     trade_date,
--     ticker,
--     SUM(CASE WHEN revision_type = 'UP' THEN 1 ELSE 0 END) AS n_up,
--     SUM(CASE WHEN revision_type = 'DOWN' THEN 1 ELSE 0 END) AS n_down,
--     (SUM(CASE WHEN revision_type = 'UP' THEN 1 ELSE 0 END)
--      - SUM(CASE WHEN revision_type = 'DOWN' THEN 1 ELSE 0 END)) * 1.0 / NULLIF(COUNT(*), 0) AS net_revision_breadth
-- FROM next_q
-- GROUP BY 1,2;
'''


### SQL Step 4.5 - Analyst dispersion as of each date


For each month-end and ticker, compute the standard deviation of broker EPS estimates for the next quarter.
Return:
- trade_date
- ticker
- fiscal_q_end
- eps_dispersion


In [ ]:

# TODO: write SQL

sql_4_5 = '''
-- SOLUTION:
-- WITH month_ends AS (
--     SELECT trade_date, ticker
--     FROM (
--         SELECT
--             trade_date,
--             ticker,
--             ROW_NUMBER() OVER (
--                 PARTITION BY ticker, DATE_TRUNC('month', trade_date)
--                 ORDER BY trade_date DESC
--             ) AS rn
--         FROM prices_daily
--     ) x
--     WHERE rn = 1
-- ),
-- latest_est AS (
--     SELECT
--         m.trade_date,
--         m.ticker,
--         a.broker_id,
--         a.fiscal_q_end,
--         a.estimate_value,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, m.ticker, a.broker_id, a.fiscal_q_end
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM month_ends m
--     JOIN analyst_eps_detail a
--       ON a.ticker = m.ticker
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
--      AND a.estimate_date <= m.trade_date
-- ),
-- next_q AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY trade_date, ticker, broker_id
--                 ORDER BY fiscal_q_end
--             ) AS fiscal_rank
--         FROM latest_est
--         WHERE rn = 1
--           AND fiscal_q_end >= trade_date
--     ) z
--     WHERE fiscal_rank = 1
-- )
-- SELECT
--     trade_date,
--     ticker,
--     fiscal_q_end,
--     STDDEV_SAMP(estimate_value) AS eps_dispersion
-- FROM next_q
-- GROUP BY 1,2,3;
'''


### SQL Step 4.6 - Count of active analysts covering each stock


As of each month-end and ticker, count the number of distinct brokers contributing to the next-quarter EPS consensus.
Return:
- trade_date
- ticker
- n_analysts


In [ ]:

# TODO: write SQL

sql_4_6 = '''
-- SOLUTION:
-- WITH month_ends AS (
--     SELECT trade_date, ticker
--     FROM (
--         SELECT
--             trade_date,
--             ticker,
--             ROW_NUMBER() OVER (
--                 PARTITION BY ticker, DATE_TRUNC('month', trade_date)
--                 ORDER BY trade_date DESC
--             ) AS rn
--         FROM prices_daily
--     ) x
--     WHERE rn = 1
-- ),
-- latest_est AS (
--     SELECT
--         m.trade_date,
--         m.ticker,
--         a.broker_id,
--         a.fiscal_q_end,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, m.ticker, a.broker_id, a.fiscal_q_end
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM month_ends m
--     JOIN analyst_eps_detail a
--       ON a.ticker = m.ticker
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
--      AND a.estimate_date <= m.trade_date
-- ),
-- next_q AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY trade_date, ticker, broker_id
--                 ORDER BY fiscal_q_end
--             ) AS fiscal_rank
--         FROM latest_est
--         WHERE rn = 1
--           AND fiscal_q_end >= trade_date
--     ) z
--     WHERE fiscal_rank = 1
-- )
-- SELECT
--     trade_date,
--     ticker,
--     COUNT(DISTINCT broker_id) AS n_analysts
-- FROM next_q
-- GROUP BY 1,2;
'''


### SQL Step 4.7 - Earnings surprise using prior-day consensus


For each earnings announcement, compute surprise:

`surprise = actual_eps - consensus_eps_prior_day`

where `consensus_eps_prior_day` is the consensus from the latest broker estimates available **strictly before** the earnings announcement date.

Return:
- ticker
- fiscal_q_end
- announce_date
- actual_eps
- consensus_eps_prior_day
- surprise


In [ ]:

# TODO: write SQL

sql_4_7 = '''
-- SOLUTION:
-- WITH broker_prior AS (
--     SELECT
--         e.ticker,
--         e.fiscal_q_end,
--         e.announce_date,
--         e.actual_eps,
--         a.broker_id,
--         a.estimate_value,
--         ROW_NUMBER() OVER (
--             PARTITION BY e.ticker, e.fiscal_q_end, e.announce_date, a.broker_id
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM earnings_actuals e
--     JOIN analyst_eps_detail a
--       ON a.ticker = e.ticker
--      AND a.fiscal_q_end = e.fiscal_q_end
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
--      AND a.estimate_date < e.announce_date
-- )
-- SELECT
--     ticker,
--     fiscal_q_end,
--     announce_date,
--     MAX(actual_eps) AS actual_eps,
--     AVG(CASE WHEN rn = 1 THEN estimate_value END) AS consensus_eps_prior_day,
--     MAX(actual_eps) - AVG(CASE WHEN rn = 1 THEN estimate_value END) AS surprise
-- FROM broker_prior
-- GROUP BY 1,2,3;
'''


### SQL Step 4.8 - Join surprises to post-earnings returns


Join the earnings surprise output to `prices_daily` to compute:
- 1-day post-earnings return
- 5-day post-earnings return

Use the first trade date on or after `announce_date`.


In [ ]:

# TODO: write SQL

sql_4_8 = '''
-- SOLUTION:
-- WITH surprise AS (
--     SELECT
--         ticker,
--         fiscal_q_end,
--         announce_date,
--         actual_eps,
--         consensus_eps_prior_day,
--         actual_eps - consensus_eps_prior_day AS surprise
--     FROM your_surprise_cte_or_table
-- ),
-- aligned_trade AS (
--     SELECT
--         s.*,
--         p.trade_date,
--         p.ret_1d,
--         ROW_NUMBER() OVER (
--             PARTITION BY s.ticker, s.fiscal_q_end, s.announce_date
--             ORDER BY p.trade_date
--         ) AS rn
--     FROM surprise s
--     JOIN prices_daily p
--       ON p.ticker = s.ticker
--      AND p.trade_date >= s.announce_date
-- ),
-- evt0 AS (
--     SELECT *
--     FROM aligned_trade
--     WHERE rn = 1
-- )
-- SELECT
--     e.ticker,
--     e.fiscal_q_end,
--     e.announce_date,
--     e.surprise,
--     p1.ret_1d AS post_ret_1d,
--     EXP(SUM(LN(1 + p5.ret_1d))) - 1 AS post_ret_5d
-- FROM evt0 e
-- LEFT JOIN prices_daily p1
--   ON p1.ticker = e.ticker
--  AND p1.trade_date = e.trade_date
-- LEFT JOIN prices_daily p5
--   ON p5.ticker = e.ticker
--  AND p5.trade_date BETWEEN e.trade_date AND e.trade_date + INTERVAL '7' DAY
-- GROUP BY 1,2,3,4,5;
'''


### SQL Step 4.9 - PIT fundamentals join to daily prices


For each `(trade_date, ticker)` in `prices_daily`, attach the latest announced quarterly fundamentals as of that date.

Return:
- trade_date
- ticker
- close_price
- latest_announce_date
- sales
- assets
- book_equity
- net_income


In [ ]:

# TODO: write SQL

sql_4_9 = '''
-- SOLUTION:
-- WITH pit_fund AS (
--     SELECT
--         p.trade_date,
--         p.ticker,
--         p.close_price,
--         f.announce_date,
--         f.sales,
--         f.assets,
--         f.book_equity,
--         f.net_income,
--         ROW_NUMBER() OVER (
--             PARTITION BY p.trade_date, p.ticker
--             ORDER BY f.announce_date DESC
--         ) AS rn
--     FROM prices_daily p
--     LEFT JOIN fundamentals_quarterly f
--       ON p.ticker = f.ticker
--      AND f.announce_date <= p.trade_date
-- )
-- SELECT
--     trade_date,
--     ticker,
--     close_price,
--     announce_date AS latest_announce_date,
--     sales,
--     assets,
--     book_equity,
--     net_income
-- FROM pit_fund
-- WHERE rn = 1;
'''


### SQL Step 4.10 - Build a monthly research feature table


Create a monthly table with one row per `(trade_date, ticker)` where `trade_date` is the last trading day of the month, including:
- close_price
- market_cap
- 21-day average dollar volume
- next-quarter EPS consensus
- 30-day EPS revision
- EPS dispersion
- number of analysts
- PIT book-to-market (using latest announced fundamentals)

This is very close to what an actual alpha-feature pull might look like.


In [ ]:

# TODO: write SQL

sql_4_10 = '''
-- SOLUTION:
-- WITH month_ends AS (
--     SELECT *
--     FROM (
--         SELECT
--             trade_date,
--             ticker,
--             close_price,
--             market_cap,
--             dollar_volume,
--             ROW_NUMBER() OVER (
--                 PARTITION BY ticker, DATE_TRUNC('month', trade_date)
--                 ORDER BY trade_date DESC
--             ) AS rn
--         FROM prices_daily
--     ) x
--     WHERE rn = 1
-- ),
-- adv21 AS (
--     SELECT
--         trade_date,
--         ticker,
--         AVG(dollar_volume) OVER (
--             PARTITION BY ticker
--             ORDER BY trade_date
--             ROWS BETWEEN 21 PRECEDING AND 1 PRECEDING
--         ) AS adv_21d
--     FROM prices_daily
-- ),
-- pit_fund AS (
--     SELECT
--         m.trade_date,
--         m.ticker,
--         f.book_equity,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, m.ticker
--             ORDER BY f.announce_date DESC
--         ) AS rn
--     FROM month_ends m
--     LEFT JOIN fundamentals_quarterly f
--       ON f.ticker = m.ticker
--      AND f.announce_date <= m.trade_date
-- ),
-- next_q_consensus AS (
--     -- Replace with result from Drill 4.2
--     SELECT trade_date, ticker, fiscal_q_end, consensus_eps
--     FROM your_next_q_consensus_cte_or_table
-- ),
-- rev30 AS (
--     -- Replace with result from Drill 4.3
--     SELECT trade_date, ticker, revision_30d
--     FROM your_revision_30d_cte_or_table
-- ),
-- disp AS (
--     -- Replace with result from Drill 4.5
--     SELECT trade_date, ticker, eps_dispersion
--     FROM your_dispersion_cte_or_table
-- ),
-- cov AS (
--     -- Replace with result from Drill 4.6
--     SELECT trade_date, ticker, n_analysts
--     FROM your_coverage_cte_or_table
-- )
-- SELECT
--     m.trade_date,
--     m.ticker,
--     m.close_price,
--     m.market_cap,
--     a.adv_21d,
--     c.consensus_eps,
--     r.revision_30d,
--     d.eps_dispersion,
--     v.n_analysts,
--     pf.book_equity / NULLIF(m.market_cap, 0) AS book_to_market
-- FROM month_ends m
-- LEFT JOIN adv21 a
--   ON a.trade_date = m.trade_date
--  AND a.ticker = m.ticker
-- LEFT JOIN next_q_consensus c
--   ON c.trade_date = m.trade_date
--  AND c.ticker = m.ticker
-- LEFT JOIN rev30 r
--   ON r.trade_date = m.trade_date
--  AND r.ticker = m.ticker
-- LEFT JOIN disp d
--   ON d.trade_date = m.trade_date
--  AND d.ticker = m.ticker
-- LEFT JOIN cov v
--   ON v.trade_date = m.trade_date
--  AND v.ticker = m.ticker
-- LEFT JOIN pit_fund pf
--   ON pf.trade_date = m.trade_date
--  AND pf.ticker = m.ticker
--  AND pf.rn = 1;
'''


### SQL Step 4.11 - Detect stale consensus data


For each month-end and ticker, compute:
- date of the most recent broker estimate contributing to next-quarter consensus
- days since latest estimate
- stale flag if > 45 days

This is useful for quality control on estimate data.


In [ ]:

# TODO: write SQL

sql_4_11 = '''
-- SOLUTION:
-- WITH month_ends AS (
--     SELECT trade_date, ticker
--     FROM (
--         SELECT
--             trade_date,
--             ticker,
--             ROW_NUMBER() OVER (
--                 PARTITION BY ticker, DATE_TRUNC('month', trade_date)
--                 ORDER BY trade_date DESC
--             ) AS rn
--         FROM prices_daily
--     ) x
--     WHERE rn = 1
-- ),
-- latest_est AS (
--     SELECT
--         m.trade_date,
--         m.ticker,
--         a.broker_id,
--         a.fiscal_q_end,
--         a.estimate_date,
--         ROW_NUMBER() OVER (
--             PARTITION BY m.trade_date, m.ticker, a.broker_id, a.fiscal_q_end
--             ORDER BY a.estimate_date DESC
--         ) AS rn
--     FROM month_ends m
--     JOIN analyst_eps_detail a
--       ON a.ticker = m.ticker
--      AND a.metric = 'EPS'
--      AND a.fiscal_period_type = 'Q'
--      AND a.is_active = 1
--      AND a.estimate_date <= m.trade_date
-- ),
-- next_q AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY trade_date, ticker, broker_id
--                 ORDER BY fiscal_q_end
--             ) AS fiscal_rank
--         FROM latest_est
--         WHERE rn = 1
--           AND fiscal_q_end >= trade_date
--     ) z
--     WHERE fiscal_rank = 1
-- )
-- SELECT
--     trade_date,
--     ticker,
--     MAX(estimate_date) AS latest_estimate_date,
--     trade_date - MAX(estimate_date) AS days_since_latest_estimate,
--     CASE WHEN trade_date - MAX(estimate_date) > 45 THEN 1 ELSE 0 END AS stale_consensus_flag
-- FROM next_q
-- GROUP BY 1,2;
'''


### SQL Step 4.12 - Broker-level revision momentum


For each `(ticker, broker_id, fiscal_q_end)`, compute:
- latest estimate
- previous estimate
- latest minus previous
- number of estimate updates in the last 90 days

This is useful when building broker-behavior features.


In [ ]:

# TODO: write SQL

sql_4_12 = '''
-- SOLUTION:
-- WITH hist AS (
--     SELECT
--         ticker,
--         broker_id,
--         fiscal_q_end,
--         estimate_date,
--         estimate_value,
--         LAG(estimate_value) OVER (
--             PARTITION BY ticker, broker_id, fiscal_q_end
--             ORDER BY estimate_date
--         ) AS prev_estimate_value,
--         ROW_NUMBER() OVER (
--             PARTITION BY ticker, broker_id, fiscal_q_end
--             ORDER BY estimate_date DESC
--         ) AS rn_latest,
--         COUNT(*) OVER (
--             PARTITION BY ticker, broker_id, fiscal_q_end
--         ) AS total_updates
--     FROM analyst_eps_detail
--     WHERE metric = 'EPS'
--       AND fiscal_period_type = 'Q'
--       AND is_active = 1
-- )
-- SELECT
--     ticker,
--     broker_id,
--     fiscal_q_end,
--     estimate_date AS latest_estimate_date,
--     estimate_value AS latest_estimate_value,
--     prev_estimate_value,
--     estimate_value - prev_estimate_value AS latest_revision,
--     total_updates
-- FROM hist
-- WHERE rn_latest = 1;
'''
